# RASP programs and compiler schedules

Define the twelve experiments, check decoded outputs against RASP, and inspect
where Tracr allocates each sequence variable in the residual stream.
Factories live in `raspformer/programs.py`; compilation and checks live in
`raspformer/compiler.py`. Run these cells in order with the `.venv` kernel.

In [1]:
import os
from pathlib import Path

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    root
    for root in (start, *start.parents)
    if (root / "raspformer").is_dir() and (root / "notebooks").is_dir()
)
os.chdir(PROJECT_ROOT)

import json

import pandas as pd
from IPython.display import Markdown, display

from raspformer.compiler import CompilerConfig, run_study, schedule_table, summary_table
from raspformer.programs import PROGRAM_DESCRIPTIONS, build_programs

## Configure the comparison

Use the same integer vocabulary, sequence limit, noncausal compiler, and
exactness setting for all programs. The five inputs cover repetition,
singleton sequences, negative values, and maximum length.

In [2]:
config = CompilerConfig(
    vocab=tuple(range(-2, 6)),
    max_seq_len=6,
    causal=False,
    bos="BOS",
    pad="PAD",
    mlp_exactness=100,
    rel_tol=1e-6,
    abs_tol=1e-6,
)
samples = [
    [1, 2, 1],
    [-2, 0, 3, 1],
    [5],
    [0, 0, 0, 0],
    [5, 4, 3, 2, 1, 0],
]

OUTPUT_ROOT = PROJECT_ROOT / "results" / "programs"
FOCUS_PROGRAM = "P05_histogram"

In [3]:
PROGRAMS = build_programs(config.max_seq_len)
display(
    pd.DataFrame(
        [{"program": name, "behavior": PROGRAM_DESCRIPTIONS[name]} for name in PROGRAMS]
    )
)

,program,behavior
0,P01_absolute,Absolute value of each token.
1,P02_index_parity,Zero-based position modulo two.
2,P03_increment_by_index,Token plus its zero-based position.
3,P04_first_element,First token repeated at every position.
4,P05_histogram,Occurrences of each token in the full sequence.
5,P06_count_greater_than,Number of tokens strictly greater than the cur...
6,P07_sum_with_next,Token plus successor; the final token adds its...
7,P08_pairwise_sum,Token plus predecessor; the first wraps to the...
8,P09_check_increasing,One everywhere iff the sequence is nondecreasing.
9,P10_rotate_left,Cyclic left rotation by one position.


## Compile and validate

A run succeeds only if each decoded sequence matches the RASP evaluator.
Every run gets its own directory; failures retain their status and evidence.

In [4]:
result = run_study(PROGRAMS, config, samples, OUTPUT_ROOT)
evidence = result["evidence"]
print(f"Saved evidence: {result['run_dir']}")
display(Markdown(summary_table(evidence["programs"])))

P01_absolute: 1 layers; 5/5 checks


P02_index_parity: 1 layers; 5/5 checks


P03_increment_by_index: 1 layers; 5/5 checks


P04_first_element: 1 layers; 5/5 checks


P05_histogram: 1 layers; 5/5 checks
P06_count_greater_than: 1 layers; 5/5 checks


...

P09_check_increasing: 3 layers; 5/5 checks


P10_rotate_left: 3 layers; 5/5 checks


P11_check_palindrome: 6 layers; 5/5 checks


P12_sorting: 3 layers; 5/5 checks
Saved evidence: /Users/raghul/RaspFormer/results/programs/20261006T062028_d1aeda94


| Program | Layers | Residual width | Computed variables | Checks passed |
| --- | ---: | ---: | ---: | ---: |
| P01_absolute | 1 | 22 | 1 | 5 |
| P02_index_parity | 1 | 19 | 1 | 5 |
| P03_increment_by_index | 1 | 30 | 1 | 5 |
| P04_first_element | 1 | 25 | 1 | 5 |
| P05_histogram | 1 | 25 | 1 | 5 |
| P06_count_greater_than | 1 | 25 | 1 | 5 |
| P07_sum_with_next | 3 | 55 | 4 | 5 |
| P08_pairwise_sum | 3 | 54 | 4 | 5 |
| P09_check_increasing | 3 | 37 | 4 | 5 |
| P10_rotate_left | 3 | 39 | 3 | 5 |
| P11_check_palindrome | 6 | 69 | 7 | 5 |
| P12_sorting | 3 | 81 | 3 | 5 |

## Inspect a writer schedule

These are compiler allocations, not measured activation onset. A
`SelectorWidth` attention/MLP pair writes its sequence variable after the MLP.
Change `FOCUS_PROGRAM` to inspect another program.

In [5]:
display(Markdown(schedule_table(evidence["variables"], FOCUS_PROGRAM)))
checks = pd.DataFrame(evidence["checks"])
display(checks[checks.program == FOCUS_PROGRAM].drop(columns="program"))

| Variable | Lanes | First writer |
| --- | ---: | --- |
| tokens | 15–22 | input embedding |
| histogram_6 | 0–6 | after mlp1 |

,tokens,expected,actual
20,"[1, 2, 1]","[2, 1, 2]","[2, 1, 2]"
21,"[-2, 0, 3, 1]","[1, 1, 1, 1]","[1, 1, 1, 1]"
22,[5],[1],[1]
23,"[0, 0, 0, 0]","[4, 4, 4, 4]","[4, 4, 4, 4]"
24,"[5, 4, 3, 2, 1, 0]","[1, 1, 1, 1, 1, 1]","[1, 1, 1, 1, 1, 1]"


In [6]:
saved = json.loads((result["run_dir"] / "step1_ground_truth_schedule.json").read_text())
assert saved["status"] == "completed"
print(
    f"{len(saved['programs'])} programs; {len(saved['checks'])} sampled checks passed."
)
print(f"Devices: {saved['settings']['devices']}")
print(f"Tracr revision: {saved['settings']['tracr_commit']}")

12 programs; 60 sampled checks passed.
Devices: ['cpu:0']
Tracr revision: 9ce2b8c82b6ba10e62e86cf6f390e7536d4fd2cd


The sampled checks establish correctness for these inputs. They do not prove
correctness for every sequence. Continue with `compile.ipynb` to compare
manual Craft traces with the assembled transformer.